# **Importing the Libraries**

In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [3]:
tf.__version__

'2.19.0'

# **Data Preprocessing**

**Importing Data from Drive**

In [4]:
from google.colab import drive
drive.mount('/content/drive')

data_dir = "/content/drive/MyDrive/Garbage classification"

Mounted at /content/drive


**Verifying Directory Contents**

In [5]:
import os

print(os.listdir(data_dir))

['glass', 'trash', 'paper', 'plastic', 'metal', 'cardboard']


**Data Wrangling and Preprocessing using TensorFlow**

In [6]:
import tensorflow as tf

img_size = (224, 224)
batch_size = 32

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

Found 2526 files belonging to 6 classes.
Using 2021 files for training.
Found 2526 files belonging to 6 classes.
Using 505 files for validation.


In [7]:
class_names = train_ds.class_names
print(class_names)

['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


**Normalization of Data**

In [8]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

**Data Optimization**

In [9]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [10]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

## **CNN Model Building**

In [11]:
cnn = tf.keras.models.Sequential()

#Layer 1
cnn.add(tf.keras.layers.Conv2D(32, 3, activation='relu', input_shape=(224,224,3)))
cnn.add(tf.keras.layers.MaxPooling2D(2,2))


#Layer 2
cnn.add(tf.keras.layers.Conv2D(64, 3, activation='relu'))
cnn.add(tf.keras.layers.MaxPooling2D(2,2))


#Layer 3
cnn.add(tf.keras.layers.Conv2D(128, 3, activation='relu'))
cnn.add(tf.keras.layers.MaxPooling2D(2,2))


#Flatten
cnn.add(tf.keras.layers.Flatten())



#Dense and Dropout
cnn.add(tf.keras.layers.Dense(128, activation='relu'))
cnn.add(tf.keras.layers.Dropout(0.5))



#Output
cnn.add(tf.keras.layers.Dense(6, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


**Data Augmentation & Early Stopping**

In [12]:
train_ds = train_ds.map(lambda x, y: (data_augmentation(x), y))

In [13]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

**Compiling CNN**

In [14]:
cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# **Training CNN**

In [15]:
cnn.fit(train_ds, validation_data=val_ds, epochs=25, callbacks=[early_stop])

Epoch 1/25
64/64 ━━━━━━━━━━━━━━━━━━━━ 557s 5s/step - accuracy: 0.2806 - loss: 1.7662 - val_accuracy: 0.4238 - val_loss: 1.4746
Epoch 2/25
64/64 ━━━━━━━━━━━━━━━━━━━━ 247s 4s/step - accuracy: 0.4265 - loss: 1.4298 - val_accuracy: 0.4416 - val_loss: 1.3733
Epoch 3/25
64/64 ━━━━━━━━━━━━━━━━━━━━ 246s 4s/step - accuracy: 0.4567 - loss: 1.3596 - val_accuracy: 0.4257 - val_loss: 1.3222
Epoch 4/25
64/64 ━━━━━━━━━━━━━━━━━━━━ 239s 4s/step - accuracy: 0.4775 - loss: 1.3132 - val_accuracy: 0.4416 - val_loss: 1.3326
Epoch 5/25
64/64 ━━━━━━━━━━━━━━━━━━━━ 240s 4s/step - accuracy: 0.4998 - loss: 1.2964 - val_accuracy: 0.5069 - val_loss: 1.2387
Epoch 6/25
64/64 ━━━━━━━━━━━━━━━━━━━━ 237s 4s/step - accuracy: 0.5191 - loss: 1.2335 - val_accuracy: 0.5248 - val_loss: 1.2131
Epoch 7/25
64/64 ━━━━━━━━━━━━━━━━━━━━ 239s 4s/step - accuracy: 0.5136 - loss: 1.2321 - val_accuracy: 0.4851 - val_loss: 1.1962
Epoch 8/25
64/64 ━━━━━━━━━━━━━━━━━━━━ 236s 4s/step - accuracy: 0.5369 - loss: 1.1859 - val_accuracy: 0.5010 - v

# **Transfer Learning Model**

**Importing Libraries and DataSet**

In [34]:
import tensorflow as tf
import os
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Data Preprocessing**

In [35]:
data_dir = "/content/drive/MyDrive/Garbage classification"
print(os.listdir(data_dir))

['glass', 'trash', 'paper', 'plastic', 'metal', 'cardboard']


In [36]:
img_size = (224, 224)
batch_size = 32

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

Found 2526 files belonging to 6 classes.
Using 2021 files for training.
Found 2526 files belonging to 6 classes.
Using 505 files for validation.


In [37]:
class_names = train_ds.class_names
print(class_names)

['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


In [38]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y))
val_ds = val_ds.map(lambda x, y: (preprocess_input(x), y))

**Data Augmentation & Optimization**

In [39]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [40]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

**Importing the Model**

In [41]:
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

In [42]:
model = tf.keras.Sequential([
    data_augmentation,
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(6, activation='softmax')
])

In [43]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [44]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [45]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop]
)

Epoch 1/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 164s 2s/step - accuracy: 0.2637 - loss: 2.2560 - val_accuracy: 0.4356 - val_loss: 1.4680
Epoch 2/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 123s 2s/step - accuracy: 0.4542 - loss: 1.5696 - val_accuracy: 0.5861 - val_loss: 1.1591
Epoch 3/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 142s 2s/step - accuracy: 0.5344 - loss: 1.3197 - val_accuracy: 0.6257 - val_loss: 1.0022
Epoch 4/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 123s 2s/step - accuracy: 0.5854 - loss: 1.1966 - val_accuracy: 0.6396 - val_loss: 0.9360
Epoch 5/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 124s 2s/step - accuracy: 0.6141 - loss: 1.0705 - val_accuracy: 0.6495 - val_loss: 0.9048
Epoch 6/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 122s 2s/step - accuracy: 0.6531 - loss: 1.0011 - val_accuracy: 0.6634 - val_loss: 0.8644
Epoch 7/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 164s 2s/step - accuracy: 0.6561 - loss: 0.9483 - val_accuracy: 0.6752 - val_loss: 0.8365
Epoch 8/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 124s 2s/step - accuracy: 0.6754 - loss: 0.9221 - val_accuracy: 0.6851 - v

In [46]:
base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False

In [47]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

**Final Training and Accuracy**

In [48]:
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Epoch 1/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 187s 3s/step - accuracy: 0.6091 - loss: 1.1208 - val_accuracy: 0.7069 - val_loss: 0.7883
Epoch 2/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 169s 3s/step - accuracy: 0.6195 - loss: 1.0679 - val_accuracy: 0.7050 - val_loss: 0.7922
Epoch 3/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 201s 3s/step - accuracy: 0.6442 - loss: 1.0046 - val_accuracy: 0.7129 - val_loss: 0.7948
Epoch 4/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 167s 3s/step - accuracy: 0.6774 - loss: 0.9239 - val_accuracy: 0.7129 - val_loss: 0.7886
Epoch 5/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 168s 3s/step - accuracy: 0.6710 - loss: 0.8731 - val_accuracy: 0.7109 - val_loss: 0.7839
